# 🎨 TRELLIS: Image to 3D GLB Converter

Convert images to high-quality 3D models using Microsoft's TRELLIS!

## Before You Start

1. **Enable GPU**: Go to `Runtime` → `Change runtime type` → Select `T4 GPU` or `A100 GPU`
2. **Check GPU**: Run the cell below to verify GPU is available
3. **Upload Images**: You'll be able to upload images later in the notebook

---

## ✅ Step 1: Check GPU Availability

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("\n⚠️ WARNING: No GPU detected! Please enable GPU in Runtime settings.")

## 📦 Step 2: Install Dependencies

This will take 5-10 minutes on first run. Subsequent runs will be faster.

In [ ]:
import os
import sys
from pathlib import Path

# Install system dependencies
print("Installing system dependencies...")
!apt-get update -qq
!apt-get install -y -qq git git-lfs

# Clone TRELLIS if not already cloned
TRELLIS_DIR = Path("/content/TRELLIS")
if not TRELLIS_DIR.exists():
    print("\nCloning TRELLIS repository...")
    !git clone --recursive https://github.com/microsoft/TRELLIS.git /content/TRELLIS
else:
    print("\nTRELLIS already cloned.")

# Install TRELLIS dependencies
os.chdir("/content/TRELLIS")
print("\nInstalling TRELLIS dependencies...")
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q xformers
!pip install -q -r requirements.txt
!pip install -q huggingface_hub

# Install trellis-local-tool dependencies
print("\nInstalling tool dependencies...")
!pip install -q click pillow pyyaml tqdm rich trimesh pygltflib

# Add TRELLIS to Python path
if str(TRELLIS_DIR) not in sys.path:
    sys.path.insert(0, str(TRELLIS_DIR))

print("\n✅ Installation complete!")

## 🚀 Step 3: Load TRELLIS Model

This downloads the model (~5GB) and loads it into GPU memory. Only needs to run once per session.

In [ ]:
import torch
from trellis.pipelines import TrellisImageTo3DPipeline
from PIL import Image
import numpy as np

# Configuration
MODEL_NAME = "JeffreyXiang/TRELLIS-image-large"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model: {MODEL_NAME}")
print(f"Device: {DEVICE}")
print("This may take a few minutes on first run...\n")

# Load pipeline
pipeline = TrellisImageTo3DPipeline.from_pretrained(MODEL_NAME)
pipeline = pipeline.to(DEVICE)

print("\n✅ Model loaded successfully!")
print(f"GPU Memory Used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 🛠️ Step 4: Helper Functions

In [ ]:
import trimesh
from pathlib import Path
from IPython.display import display, Image as IPImage, FileLink
import matplotlib.pyplot as plt

def process_image_to_3d(image_path, output_path="output.glb", seed=1, show_preview=True):
    """
    Convert an image to a 3D GLB file.
    
    Args:
        image_path: Path to input image
        output_path: Path for output GLB file
        seed: Random seed for reproducibility
        show_preview: Show input image preview
    
    Returns:
        Path to generated GLB file
    """
    # Load image
    print(f"Loading image: {image_path}")
    image = Image.open(image_path)
    
    if image.mode != "RGB":
        image = image.convert("RGB")
    
    # Show preview
    if show_preview:
        plt.figure(figsize=(8, 8))
        plt.imshow(image)
        plt.axis('off')
        plt.title(f"Input Image: {Path(image_path).name}")
        plt.show()
    
    print(f"Image size: {image.size[0]}x{image.size[1]}")
    
    # Set seed
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    # Run TRELLIS
    print("\nRunning TRELLIS inference...")
    print("This may take 30-60 seconds...")
    
    outputs = pipeline.run(image, seed=seed)
    
    print("✅ Inference complete!")
    
    # Extract and export mesh
    print(f"\nExporting to GLB: {output_path}")
    
    # Get mesh data from outputs
    # Note: The exact structure depends on TRELLIS version
    # This is a placeholder - you may need to adjust based on actual output structure
    try:
        # Try to get mesh directly
        if hasattr(outputs, 'mesh'):
            mesh_data = outputs.mesh
        elif 'mesh' in outputs:
            mesh_data = outputs['mesh']
        else:
            # Extract from radiance field
            print("Extracting mesh from outputs...")
            mesh_data = outputs
        
        # Export using TRELLIS built-in export if available
        if hasattr(mesh_data, 'export'):
            mesh_data.export(output_path)
        else:
            # Manual export
            vertices = mesh_data.get('vertices') or mesh_data.get('verts')
            faces = mesh_data.get('faces') or mesh_data.get('tris')
            
            if vertices is not None and faces is not None:
                mesh = trimesh.Trimesh(
                    vertices=vertices.cpu().numpy() if torch.is_tensor(vertices) else vertices,
                    faces=faces.cpu().numpy() if torch.is_tensor(faces) else faces
                )
                mesh.export(output_path, file_type='glb')
            else:
                print("⚠️ Could not extract mesh data. Available keys:", list(outputs.keys()) if isinstance(outputs, dict) else dir(outputs))
                return None
        
        # Check file size
        file_size = Path(output_path).stat().st_size / 1e6
        print(f"\n✅ Success!")
        print(f"Output: {output_path}")
        print(f"Size: {file_size:.2f} MB")
        
        return output_path
        
    except Exception as e:
        print(f"❌ Export failed: {e}")
        print("\nDebug info:")
        print(f"Output type: {type(outputs)}")
        if isinstance(outputs, dict):
            print(f"Output keys: {list(outputs.keys())}")
        return None

def batch_process(image_dir, output_dir="/content/outputs", seed=1):
    """
    Process all images in a directory.
    
    Args:
        image_dir: Directory containing images
        output_dir: Directory for output GLB files
        seed: Random seed
    """
    import glob
    
    # Create output directory
    Path(output_dir).mkdir(exist_ok=True)
    
    # Find images
    extensions = ['*.jpg', '*.jpeg', '*.png', '*.webp']
    images = []
    for ext in extensions:
        images.extend(glob.glob(f"{image_dir}/{ext}"))
        images.extend(glob.glob(f"{image_dir}/{ext.upper()}"))
    
    if not images:
        print(f"No images found in {image_dir}")
        return
    
    print(f"Found {len(images)} images\n")
    
    results = []
    for i, img_path in enumerate(images, 1):
        print(f"\n{'='*60}")
        print(f"[{i}/{len(images)}] Processing: {Path(img_path).name}")
        print(f"{'='*60}")
        
        output_name = Path(img_path).stem + ".glb"
        output_path = Path(output_dir) / output_name
        
        try:
            result = process_image_to_3d(img_path, str(output_path), seed=seed, show_preview=False)
            if result:
                results.append(result)
        except Exception as e:
            print(f"❌ Failed: {e}")
            continue
    
    print(f"\n\n{'='*60}")
    print(f"Batch Complete: {len(results)}/{len(images)} successful")
    print(f"{'='*60}")
    print(f"Output directory: {output_dir}")

print("✅ Helper functions loaded!")

## 📤 Step 5: Upload Images

Upload one or more images to convert.

In [ ]:
from google.colab import files
import shutil

# Create input directory
input_dir = Path("/content/inputs")
input_dir.mkdir(exist_ok=True)

print("📤 Click 'Choose Files' to upload images...\n")
uploaded = files.upload()

# Move uploaded files to input directory
for filename in uploaded.keys():
    shutil.move(filename, input_dir / filename)
    print(f"✅ Uploaded: {filename}")

print(f"\nAll files saved to: {input_dir}")

## 🎨 Step 6: Convert Single Image

Convert a single image to 3D.

In [ ]:
# Configuration
IMAGE_PATH = "/content/inputs/your-image.jpg"  # Change this to your uploaded image name
OUTPUT_PATH = "/content/output.glb"
SEED = 1  # Change for different results

# List available images
print("Available images:")
for img in input_dir.glob("*"):
    if img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
        print(f"  - {img.name}")

print("\n" + "="*60)

# Convert
result = process_image_to_3d(IMAGE_PATH, OUTPUT_PATH, seed=SEED)

if result:
    print("\n📥 Download your 3D model:")
    display(FileLink(result))

## 🔄 Step 7: Batch Convert (Optional)

Convert all uploaded images at once.

In [ ]:
OUTPUT_DIR = "/content/outputs"
SEED = 1

# Process all images
batch_process(str(input_dir), OUTPUT_DIR, seed=SEED)

# List outputs
print("\n📥 Generated 3D models:")
for glb_file in Path(OUTPUT_DIR).glob("*.glb"):
    print(f"  - {glb_file.name} ({glb_file.stat().st_size / 1e6:.2f} MB)")
    display(FileLink(str(glb_file)))

## 📦 Step 8: Download All Outputs (Optional)

Zip and download all generated GLB files.

In [ ]:
import shutil
from google.colab import files

OUTPUT_DIR = "/content/outputs"

# Create zip archive
print("Creating zip archive...")
shutil.make_archive("/content/3d_models", 'zip', OUTPUT_DIR)

print("\n📥 Downloading all models...")
files.download("/content/3d_models.zip")

print("✅ Download complete!")

## 👁️ View Your 3D Models

You can view your GLB files using:

1. **Online Viewer**: [gltf-viewer.donmccurdy.com](https://gltf-viewer.donmccurdy.com)
2. **Blender**: File → Import → glTF 2.0
3. **Windows 3D Viewer**: Built-in Windows app
4. **Three.js**: Use GLTFLoader in your web projects
5. **Unity/Unreal**: Import as 3D asset

---

## 💡 Tips

- **Best Results**: Use clear, well-lit photos with single prominent objects
- **Resolution**: 1024x1024 or higher recommended
- **Seed**: Change seed value to get different variations
- **GPU**: T4 works fine, A100 is faster for batch processing
- **Processing Time**: ~30-60 seconds per image on T4 GPU

## 🐛 Troubleshooting

- **Out of Memory**: Restart runtime and try one image at a time
- **Model Load Fails**: Check internet connection, model download may have failed
- **Export Fails**: The TRELLIS output format may have changed, check the debug output

---

## 📚 Resources

- [TRELLIS GitHub](https://github.com/microsoft/TRELLIS)
- [TRELLIS Paper](https://arxiv.org/abs/TRELLIS)
- [GLB Format Spec](https://www.khronos.org/gltf/)

---

**Enjoy creating 3D models! 🎉**